# Construção do GTFS operacional — Salineira / Cabo Frio

Este notebook documenta, de ponta a ponta, a geração de um feed GTFS a partir de:
- planilha operacional,
- itinerários vetoriais,
- pontos de parada fornecidos.

O processo é organizado em quatro blocos:
1. leitura e auditoria dos insumos;
2. saneamento dos itinerários e registro das correções;
3. construção e validação do feed GTFS;
4. exportação dos artefatos finais e dos relatórios de QA.

O objetivo e gerar `gtfs.zip` e, também, produzir um resultado auditável, com rastreabilidade entre insumos, correções aplicadas e produto final.

## 1. Ambiente de execução

Este notebook assume a seguinte organização do repositório:

- `notebooks/`: notebook principal e script auxiliar de correção de direções;
- `src/gtfsweaver/`: biblioteca principal;
- `data/raw/`: insumos originais;
- `data/interim/`: insumos corrigidos e artefatos intermediários;
- `data/processed/`: feed GTFS final e relatórios de qualidade.

Todas as saídas desta execução serão gravadas em um diretório versionado por timestamp.

In [3]:
from datetime import datetime
from pathlib import Path
import json
import hashlib
import shutil

import geopandas as gpd
import pandas as pd

from fix_itinerarios_direction import fix_directions
#from gtfsweaver import read_protofeed_from_excel, build_feed
#from gtfsweaver.qa import build_quality_report, has_blocking_issues

In [4]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

ROOT = Path("..")  # notebook rodando dentro de notebooks/
DATA_RAW = ROOT / "data" / "raw"
DATA_INTERIM = ROOT / "data" / "interim" / RUN_ID
DATA_PROCESSED = ROOT / "data" / "processed" / RUN_ID

DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
(DATA_PROCESSED / "gtfs_tables").mkdir(parents=True, exist_ok=True)
(DATA_PROCESSED / "qa").mkdir(parents=True, exist_ok=True)

RAW_ROUTES = DATA_RAW / "itinerarios_linhas_clean.gpkg"
RAW_STOPS = DATA_RAW / "pontos_parada.gpkg"
RAW_EXCEL = DATA_RAW / "make_gtfs_template_clean.xlsx"

FIXED_ROUTES = DATA_INTERIM / "itinerarios_linhas_fixed.gpkg"
AUDIT_CSV = DATA_INTERIM / "direction_fix_audit.csv"
FIXED_EXCEL = DATA_INTERIM / "make_gtfs_template_fixed.xlsx"

GTFS_ZIP = DATA_PROCESSED / "gtfs.zip"
GTFS_TABLES = DATA_PROCESSED / "gtfs_tables"
QA_DIR = DATA_PROCESSED / "qa"
MANIFEST = DATA_PROCESSED / "build_manifest.json"

## 2. Insumos e diagnóstico inicial

Antes de construir o feed, é necessário verificar se os insumos estão coerentes.

Neste caso, os itinerários vetoriais exigem um saneamento prévio de direção e duplicidades. A planilha operacional, por sua vez, será usada como fonte de verdade para janelas de serviço e tempos totais de viagem.

In [7]:
gpd.list_layers?

Signature: gpd.list_layers(filename) -> 'pd.DataFrame'
Docstring:
List layers available in a file.

Provides an overview of layers available in a file or URL together with their
geometry types. When supported by the data source, this includes both spatial and
non-spatial layers. Non-spatial layers are indicated by the ``"geometry_type"``
column being ``None``. GeoPandas will not read such layers but they can be read into
a pd.DataFrame using :func:`pyogrio.read_dataframe`.

Parameters
----------
filename : str, path object or file-like object
    Either the absolute or relative path to the file or URL to
    be opened, or any object with a read() method (such as an open file
    or StringIO)

Returns
-------
pandas.DataFrame
    A DataFrame with columns "name" and "geometry_type" and one row per layer.
File:      c:\users\brand\anaconda3\envs\roda\lib\site-packages\geopandas\io\file.py
Type:      function

In [3]:
gdf_routes_raw = gpd.read_file(RAW_ROUTES)
gdf_stops_raw = gpd.read_file(RAW_STOPS)

xls = pd.read_excel(RAW_EXCEL, sheet_name=None, dtype=str)
agency_df = xls["agency"].copy()
routes_df = xls["routes"].copy()
holidays_df = xls.get("holidays")

In [4]:
print("Itinerários:", len(gdf_routes_raw))
print("Linhas únicas nos itinerários:", gdf_routes_raw["route_short_name"].nunique())
print("Paradas:", len(gdf_stops_raw))
print("Linhas únicas na planilha:", routes_df["route_short_name"].nunique())

display(
    gdf_routes_raw["direction"].value_counts(dropna=False).rename("count")
)
display(
    routes_df["schedule_type"].astype(str).str.lower()
    .value_counts(dropna=False).rename("count")
)

Itinerários: 46
Linhas únicas nos itinerários: 23
Paradas: 3429
Linhas únicas na planilha: 23


direction
volta    23
ida      23
Name: count, dtype: int64

schedule_type
fixed    1722
Name: count, dtype: int64

O objetivo desta etapa é caracterizar o problema de entrada e registrar a situação original dos dados.

## 3. Saneamento dos itinerários

Os itinerários vetoriais não entram diretamente no builder. Antes, eles passam por um saneamento específico, cujo papel é:

- corrigir rótulos de direção inconsistentes;
- eliminar duplicidades residuais;
- alinhar a convenção de direção da geometria com a convenção adotada na planilha operacional.

Nesta etapa, o script auxiliar retorna dois artefatos:
- `gdf_fixed`: geometria corrigida;
- `audit_df`: trilha de auditoria das decisões tomadas.

In [5]:
gdf_fixed, audit_df = fix_directions(gdf_routes_raw)

Step 1 — Relabelling 'não identificado' entries


Step 2 — Deduplicating


Step 3 — Swapping ida ↔ volta to match schedule convention

  Swapped: 40 entries (skipped 3 already-consistent routes: ['313', '346', '355'])

Step 4 — Validation

  Routes: 23
  Entries: 46 (was 46, dropped 0)
  Status: ✓ all clean


In [6]:
print("Itinerários corrigidos:", len(gdf_fixed))
print("Linhas únicas:", gdf_fixed["route_short_name"].nunique())
display(audit_df.head(20))

Itinerários corrigidos: 46
Linhas únicas: 23


,route_short_name,route_long_name,fid_original,row_index,step,action,reason,direction_original,direction_final,dist_to_term_deg,dist_to_term_km,near_terminal,n_vertices,kept,dropped
0,302,São Cristóvão / Agrisa,0,0,deduplicate,keep_unique,only candidate for direction,volta,volta,0.308198,34.209938,False,429,True,False
1,302,São Cristóvão / Agrisa,0,0,swap,swap_direction,matches schedule convention,volta,ida,0.308198,34.209938,False,429,True,False
2,302,São Cristóvão / Agrisa,1,1,deduplicate,keep_unique,only candidate for direction,ida,ida,0.000899,0.099777,True,439,True,False
3,302,São Cristóvão / Agrisa,1,1,swap,swap_direction,matches schedule convention,ida,volta,0.000899,0.099777,True,439,True,False
4,303,Avenida do Contorno / Jardim Caiçara,2,2,deduplicate,keep_unique,only candidate for direction,volta,volta,0.037494,4.161790,False,42,True,False
5,303,Avenida do Contorno / Jardim Caiçara,2,2,swap,swap_direction,matches schedule convention,volta,ida,0.037494,4.161790,False,42,True,False
6,303,Avenida do Contorno / Jardim Caiçara,3,3,deduplicate,keep_unique,only candidate for direction,ida,ida,0.000391,0.043418,True,53,True,False
7,303,Avenida do Contorno / Jardim Caiçara,3,3,swap,swap_direction,matches schedule convention,ida,volta,0.000391,0.043418,True,53,True,False
8,304,Praia do Siqueira / Forte São Mateus,4,4,deduplicate,keep_unique,only candidate for direction,volta,volta,0.035170,3.903836,False,68,True,False
9,304,Praia do Siqueira / Forte São Mateus,4,4,swap,swap_direction,matches schedule convention,volta,ida,0.035170,3.903836,False,68,True,False


In [7]:
post_summary = (
    gdf_fixed.groupby(["route_short_name", "direction"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(post_summary.sort_values("route_short_name"))

bad_routes = post_summary.loc[
    (post_summary.get("ida", 0) != 1)
    | (post_summary.get("volta", 0) != 1)
]

if not bad_routes.empty:
    raise ValueError("Ainda há linhas sem exatamente uma ida e uma volta.")

direction,route_short_name,ida,volta
0,302,1,1
1,303,1,1
2,304,1,1
3,309,1,1
4,310,1,1
5,311,1,1
6,313,1,1
7,316,1,1
8,320,1,1
9,321,1,1


O objetivo desta etapa é garantir que cada linha possua exatamente uma geometria de ida e uma de volta, ambas prontas para serem usadas na geração das shapes do GTFS.

## 4. Persistência dos insumos intermediários

Como o saneamento geométrico altera o insumo original, os arquivos corrigidos são gravados separadamente. Isso preserva a distinção entre dado bruto, dado tratado e produto final.

In [8]:
gdf_fixed.to_file(FIXED_ROUTES, driver="GPKG")
audit_df.to_csv(AUDIT_CSV, index=False)
shutil.copy2(RAW_EXCEL, FIXED_EXCEL)

WindowsPath('../data/interim/20260325_183744/make_gtfs_template_fixed.xlsx')

In [9]:
print("Geometria corrigida:", FIXED_ROUTES)
print("Auditoria:", AUDIT_CSV)
print("Planilha de trabalho:", FIXED_EXCEL)

Geometria corrigida: ..\data\interim\20260325_183744\itinerarios_linhas_fixed.gpkg
Auditoria: ..\data\interim\20260325_183744\direction_fix_audit.csv
Planilha de trabalho: ..\data\interim\20260325_183744\make_gtfs_template_fixed.xlsx


## 5. Preparação para o build

Com os itinerários corrigidos e os demais insumos consolidados, o próximo passo é construir o `ProtoFeed`, isto é, a representação intermediária normalizada que antecede a montagem das tabelas GTFS.

In [10]:
pfeed = read_protofeed_from_excel(
    xlsx_path=FIXED_EXCEL,
    routes_geo_path=FIXED_ROUTES,
    stops_geo_path=RAW_STOPS,
)

In [11]:
print("Service windows:", len(pfeed.service_windows))
print("Frequencies/resolved rows:", len(pfeed.frequencies))
print("Shapes:", len(pfeed.shapes))
print("Stops table available:", pfeed.stops is not None)

Service windows: 748
Frequencies/resolved rows: 1722
Shapes: 46
Stops table available: True


## 6. Construção do feed GTFS

Nesta execução, o feed é construído em modo proporcional. Isso significa que os tempos de percurso entre paradas são distribuídos ao longo da shape a partir do tempo total informado para cada viagem na planilha operacional.

Para esta rodada, o foco é produzir um GTFS funcional e auditável para análises em horário comercial e entre picos, com ênfase em:
- calendários corretos;
- trips consistentes;
- stop_times plausíveis.

In [12]:
BUILD_PARAMS = {
    "speed_mode": "proportional",
    "buffer": 25,
    "cluster_h3": False,
    "drop_orphans": False,
    "projected_stop_tolerance": 5.0,
    "used_stops_only": True,
}

feed = build_feed(
    pfeed,
    speed_mode=BUILD_PARAMS["speed_mode"],
    buffer=BUILD_PARAMS["buffer"],
    cluster_h3=BUILD_PARAMS["cluster_h3"],
    drop_orphans=BUILD_PARAMS["drop_orphans"],
    projected_stop_tolerance=BUILD_PARAMS["projected_stop_tolerance"],
    used_stops_only=BUILD_PARAMS["used_stops_only"],
)

In [13]:
counts = {
    "agency": len(feed.agency) if feed.agency is not None else 0,
    "routes": len(feed.routes) if feed.routes is not None else 0,
    "trips": len(feed.trips) if feed.trips is not None else 0,
    "stop_times": len(feed.stop_times) if feed.stop_times is not None else 0,
    "stops": len(feed.stops) if feed.stops is not None else 0,
    "shapes": len(feed.shapes) if feed.shapes is not None else 0,
    "frequencies": len(feed.frequencies) if feed.frequencies is not None else 0,
}
counts

{'agency': 1,
 'routes': 23,
 'trips': 1722,
 'stop_times': 130454,
 'stops': 830,
 'shapes': 8114,
 'frequencies': 0}

## 7. Validação do feed

Gerar o feed não é o fim do processo. É necessário verificar se o resultado é estruturalmente coerente e se os principais artefatos esperados foram de fato produzidos.

As verificações a seguir se concentram em:
- existência das tabelas centrais;
- consistência básica de `trips` e `stop_times`;
- produção de relatórios auxiliares de QA.

In [14]:
qa_report = build_quality_report(feed)

qa_summary = {
    name: 0 if df is None else len(df)
    for name, df in qa_report.items()
}
qa_summary

{'trips_without_stop_times': 0,
 'non_monotone_times': 0,
 'non_monotone_shape_dist': 0,
 'duplicate_trip_stop_sequence': 0}

In [15]:
for name, df in qa_report.items():
    if df is not None and len(df) > 0:
        df.to_csv(QA_DIR / f"{name}.csv", index=False)

In [16]:
if counts["routes"] == 0 or counts["trips"] == 0 or counts["stop_times"] == 0:
    raise ValueError("Feed incompleto: rotas, trips ou stop_times vazios.")

if has_blocking_issues(qa_report):
    print("Há issues bloqueantes no QA. Verifique os CSVs em:", QA_DIR)
else:
    print("QA sem issues bloqueantes.")

QA sem issues bloqueantes.


Esses relatórios não substituem uma inspeção substantiva do feed, o que será feito no [Validador Canônico](https://gtfs-validator.mobilitydata.org/), mas estabelecem um piso mínimo de confiabilidade para a exportação.

## 8. Exportação dos artefatos

A exportação final inclui não apenas o `gtfs.zip`, mas também:
- as tabelas GTFS explodidas em `.txt`;
- os relatórios de QA;
- os insumos intermediários corrigidos;
- um manifesto com parâmetros e hashes dos arquivos de entrada.

In [17]:
feed.to_file(GTFS_ZIP, ndigits=6)

In [18]:
for name in [
    "agency",
    "calendar",
    "calendar_dates",
    "routes",
    "trips",
    "stop_times",
    "stops",
    "shapes",
    "frequencies",
]:
    table = getattr(feed, name, None)
    if table is not None and len(table) > 0:
        table.to_csv(GTFS_TABLES / f"{name}.txt", index=False)

In [19]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "run_id": RUN_ID,
    "inputs": {
        "routes_raw": str(RAW_ROUTES),
        "stops_raw": str(RAW_STOPS),
        "excel_raw": str(RAW_EXCEL),
    },
    "intermediate": {
        "routes_fixed": str(FIXED_ROUTES),
        "audit_csv": str(AUDIT_CSV),
        "excel_fixed": str(FIXED_EXCEL),
    },
    "outputs": {
        "gtfs_zip": str(GTFS_ZIP),
        "gtfs_tables": str(GTFS_TABLES),
        "qa_dir": str(QA_DIR),
    },
    "hashes": {
        "routes_raw": sha256_file(RAW_ROUTES),
        "stops_raw": sha256_file(RAW_STOPS),
        "excel_raw": sha256_file(RAW_EXCEL),
    },
    "build_params": BUILD_PARAMS,
    "counts": counts,
    "qa_summary": qa_summary,
}

with open(MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

## 9. Síntese da execução

Esta execução produziu:
- um conjunto corrigido de itinerários;
- uma trilha de auditoria das correções geométricas;
- um feed GTFS com tabelas estruturais preenchidas;
- relatórios para inspeção posterior.

In [23]:
feed.stop_times.head(15)

,trip_id,stop_id,stop_sequence,arrival_time,departure_time,shape_dist_traveled,timepoint
0,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s002498,0,06:10:00,06:10:00,119.254,1
1,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s000603,1,06:10:03,06:10:03,135.326,0
2,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s002601,2,06:10:30,06:10:30,274.018,0
3,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s000375,3,06:10:38,06:10:38,315.017,0
4,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s000471,4,06:11:41,06:11:41,650.175,0
5,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s000461,5,06:12:05,06:12:05,779.770,0
6,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s000502,6,06:12:28,06:12:28,898.916,0
7,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s000482,7,06:13:08,06:13:08,1110.255,0
8,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s002163,8,06:13:30,06:13:30,1226.118,0
9,t-r353-sw_06:10:00_06:10:00_DU-06:10:00-1-0,s000832,9,06:13:57,06:13:57,1369.259,0


In [33]:
# -----------------------------------------
# Escolhas
# -----------------------------------------
route_key = "302"          # route_short_name
direction_id = 1           # 0 ou 1
use_short_name = True


In [34]:

# -----------------------------------------
# Rota
# -----------------------------------------
routes = feed.routes.copy()

if use_short_name:
    route_row = routes.loc[routes["route_short_name"] == route_key].iloc[0]
else:
    route_row = routes.loc[routes["route_id"] == route_key].iloc[0]

route_id = route_row["route_id"]

# -----------------------------------------
# Viagens da rota naquele sentido
# -----------------------------------------
trips_sel = feed.trips.loc[
    (feed.trips["route_id"] == route_id)
    & (feed.trips["direction_id"] == direction_id)
].copy()

# pega uma viagem representativa: a com mais stops
st_counts = (
    feed.stop_times.loc[feed.stop_times["trip_id"].isin(trips_sel["trip_id"])]
    .groupby("trip_id", as_index=False)
    .agg(n_stops=("stop_id", "count"))
)

trip_row = (
    trips_sel.merge(st_counts, on="trip_id", how="left")
    .fillna({"n_stops": 0})
    .sort_values("n_stops", ascending=False)
    .iloc[0]
)

trip_id = trip_row["trip_id"]
shape_id = trip_row["shape_id"]

# -----------------------------------------
# Shape
# -----------------------------------------
shape_gdf = gk.geometrize_shapes(
    feed.shapes.loc[feed.shapes["shape_id"] == shape_id].copy()
)

# -----------------------------------------
# Stops em sequência
# -----------------------------------------
st_sel = (
    feed.stop_times.loc[feed.stop_times["trip_id"] == trip_id]
    .sort_values("stop_sequence")
    .copy()
)

stops_sel = feed.stops.loc[
    feed.stops["stop_id"].isin(st_sel["stop_id"])
].copy()

stops_gdf = gk.geometrize_stops(stops_sel).merge(
    st_sel[["stop_id", "stop_sequence"]],
    on="stop_id",
    how="inner",
)

# -----------------------------------------
# Mapa
# -----------------------------------------
m = shape_gdf.explore(
    color="red" if direction_id == 0 else "blue",
    style_kwds={"weight": 5},
    tooltip=["shape_id"],
    name="shape",
    tiles="CartoDB Positron",
)

stops_gdf.explore(
    m=m,
    column="stop_sequence",
    cmap="YlOrRd" if direction_id == 0 else "PuBu",
    tooltip=["stop_sequence", "stop_id", "stop_name"],
    marker_kwds={"radius": 5},
    legend=True,
    name="stops",
)

m